# Bronze Ingestion – E-Commerce Customer Behaviour and Sales Intelligence
**Role:** Data Ingestion Engineer (Bronze). **Scope:** ingest only – no cleaning, no dedup, no filtering, no typing.

**Architecture (2 layers inside one Unity Catalog schema):**
1. **Volume (source of truth):** `landing/` = where you upload the 7 CSVs; `original/` = verified, never-modified copy (SHA-256 checked). All Bronze tables are read from `original/`.
2. **Delta tables (Bronze):** all columns are **strings**, every source row kept, plus metadata columns.
   - *Parsed* tables (Amazon, Sale Report, May-2022, P  L March 2021): normal CSV parsing.
   - *Raw-line* tables (International, Expense IIGF, Cloud Warehouse): one row per physical line (`line_number`, `raw_line`), because these files hold several tables / a text document, so CSV headers and columns would be misleading.

**Run the cells one by one, top to bottom. Only Cell 1 needs editing.**

## Cell 1 – CONFIGURATION (the only cell you edit)

In [0]:
# ============================ CHANGE ONLY THIS CELL ============================
CATALOG        = "workspace"  # <-- CHANGE 1: an existing Unity Catalog catalog where you may create a schema + volume
BRONZE_SCHEMA  = "bronze"             # <-- CHANGE 2 (optional): schema for the Bronze tables AND the Volume
VOLUME_NAME    = "raw_files"          # <-- CHANGE 3 (optional): Volume name
LANDING_FOLDER = "landing"            # folder inside the Volume where you upload the 7 CSV files
ORIGINAL_FOLDER = "original"          # folder inside the Volume for the protected copy (do not edit files here)

# SAFETY SWITCH. False = existing Bronze tables are NEVER overwritten (they are skipped and reported).
# Set to True ONLY if you deliberately want to rebuild tables that already exist.
OVERWRITE_EXISTING_TABLES = False
# ===============================================================================

import os, re, csv, hashlib, shutil, uuid
from datetime import datetime, timezone
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType

if CATALOG == "YOUR_CATALOG_NAME":
    raise ValueError("Edit CATALOG in Cell 1 first (replace YOUR_CATALOG_NAME with your real catalog name).")

VOLUME_ROOT  = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/{VOLUME_NAME}"
LANDING_DIR  = f"{VOLUME_ROOT}/{LANDING_FOLDER}"
ORIGINAL_DIR = f"{VOLUME_ROOT}/{ORIGINAL_FOLDER}"
BATCH_ID     = str(uuid.uuid4())              # identifies this notebook run
INGESTION_TS = datetime.now(timezone.utc)     # one timestamp for the whole run

def fqn(table):
    """Fully qualified table name, e.g. `my_catalog`.`bronze`.`may_2022`."""
    return f"`{CATALOG}`.`{BRONZE_SCHEMA}`.`{table}`"

print("Volume root :", VOLUME_ROOT)
print("Batch id    :", BATCH_ID)
print("Overwrite existing tables? ->", OVERWRITE_EXISTING_TABLES)

Volume root : /Volumes/workspace/bronze/raw_files
Batch id    : cac1a545-a4ec-4e42-b459-576305fee568
Overwrite existing tables? -> False


## Cell 2 – Source definitions and helper functions (do not change)   CORRECTED (replaces original Cell 2)
`exp_bytes` / `exp_count` come from your inspection report and are used only to REPORT if a file differs.
`exp_count` = data rows (excl. header) for parsed files, physical lines (incl. header) for raw-line files.

In [0]:
SOURCES = [
    {"file": "Amazon Sale Report.csv",               "table": "amazon_sale_report",             "kind": "csv", "exp_bytes": 68923428, "exp_count": 128975},
    {"file": "Sale Report.csv",                      "table": "sale_report",                    "kind": "csv", "exp_bytes": 433643,   "exp_count": 9271},
    {"file": "International sale Report.csv",        "table": "international_sale_report_raw",  "kind": "raw", "exp_bytes": 3112676,  "exp_count": 37433},
    {"file": "May-2022.csv",                         "table": "may_2022",                       "kind": "csv", "exp_bytes": 126617,   "exp_count": 1330},
    {"file": "P  L March 2021.csv",                  "table": "pl_march_2021",                  "kind": "csv", "exp_bytes": 135938,   "exp_count": 1330},  # TWO spaces between P and L
    {"file": "Expense IIGF.csv",                     "table": "expense_iigf_raw",               "kind": "raw", "exp_bytes": 496,      "exp_count": 18},
    {"file": "Cloud Warehouse Compersion Chart.csv", "table": "cloud_warehouse_comparison_raw", "kind": "raw", "exp_bytes": 4528,     "exp_count": 51},
]
METHOD_TEXT = {"csv": "CSV parsed, all columns as string", "raw": "Raw lines (line_number, raw_line)"}
COUNT_BASIS = {"csv": "data rows (excl. header)", "raw": "physical lines (incl. header)"}
META_COLS = ["_source_file", "_ingestion_timestamp", "_batch_id"]

def sha256_file(path, chunk=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

def read_csv_header(path):
    """Header names exactly as written in the file (keeps trailing spaces, 'Unnamed: 22', etc.)."""
    with open(path, newline="", encoding="utf-8") as fh:
        return next(csv.reader(fh))

def to_bronze_names(original_names):
    """Delta-safe column names: lowercase, letters/digits/underscore only. Originals are kept in _column_name_mapping."""
    out, used = [], set()
    for pos, name in enumerate(original_names):
        n = re.sub(r"[^0-9a-zA-Z]+", "_", name).strip("_").lower()
        if n == "":
            n = f"col_{pos + 1}"
        if n in used:
            n = f"{n}_{pos + 1}"
        used.add(n)
        out.append(n)
    return out

def read_raw_lines(path):
    """Return (list_of_lines, file_ends_with_newline). Nothing is stripped or translated."""
    with open(path, "r", encoding="utf-8", newline="") as fh:
        text = fh.read()
    ends_nl = text.endswith("\n")
    lines = text.split("\n")
    if ends_nl:
        lines = lines[:-1]
    return lines, ends_nl

def csv_source_profile(path):
    """Independent (non-Spark) pass. Returns (header, n_data_records, non-empty cells per column, n_ragged_rows).
    A 'ragged' row has a different number of fields than the header (must be 0, otherwise nothing is silently dropped)."""
    with open(path, newline="", encoding="utf-8") as fh:
        reader = csv.reader(fh)
        header = next(reader)
        n, ragged, nonempty = 0, 0, [0] * len(header)
        for row in reader:
            n += 1
            if len(row) != len(header):
                ragged += 1
            for i, v in enumerate(row):
                if v != "" and i < len(header):
                    nonempty[i] += 1
    return header, n, nonempty, ragged

def table_exists(table):
    return spark.catalog.tableExists(f"{CATALOG}.{BRONZE_SCHEMA}.{table}")

def write_bronze_table(df, table):
    """Create a Bronze Delta table, skip existing tables unless overwrite is enabled."""
    exists = table_exists(table)

    if exists and not OVERWRITE_EXISTING_TABLES:
        return "SKIPPED"

    # The existence check above protects existing tables when overwrite is False.
    # Delta in this workspace does not support the errorifexists write mode.
    writer = df.write.format("delta").mode("overwrite")

    if exists:
        writer = writer.option("overwriteSchema", "true")

    writer.saveAsTable(fqn(table))

    return "OVERWRITTEN" if exists else "CREATED"

def add_metadata(df, source_file):
    return (df.withColumn("_source_file", F.lit(source_file))
              .withColumn("_ingestion_timestamp", F.lit(INGESTION_TS).cast("timestamp"))
              .withColumn("_batch_id", F.lit(BATCH_ID)))

print("Helpers loaded. Sources defined:", len(SOURCES))

Helpers loaded. Sources defined: 7


## Cell 3 – Create schema, Volume and folders (safe: `IF NOT EXISTS`, never drops anything)
After this cell succeeds, **upload the 7 CSV files into the `landing` folder** (Catalog → your catalog → schema → Volume → `landing` → *Upload to this volume*).

In [0]:
try:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{BRONZE_SCHEMA}`")
    spark.sql(f"CREATE VOLUME IF NOT EXISTS `{CATALOG}`.`{BRONZE_SCHEMA}`.`{VOLUME_NAME}`")
    os.makedirs(LANDING_DIR, exist_ok=True)
    os.makedirs(ORIGINAL_DIR, exist_ok=True)
    print("OK. Schema, Volume and folders exist:")
    print("  Upload your 7 CSV files to ->", LANDING_DIR)
except Exception as e:
    raise RuntimeError(
        f"Could not create the schema/volume. Check the catalog name and that you have CREATE SCHEMA / CREATE VOLUME "
        f"permission (or ask your admin to create them). Original error: {e}")

OK. Schema, Volume and folders exist:
  Upload your 7 CSV files to -> /Volumes/workspace/bronze/raw_files/landing


## Cell 4 – Check that all 7 source files exist (stops with an error if any is missing)

In [0]:
def list_dir(d):
    return sorted(os.listdir(d)) if os.path.isdir(d) else []

landing_files, original_files = list_dir(LANDING_DIR), list_dir(ORIGINAL_DIR)
rows, missing = [], []
for s in SOURCES:
    in_l, in_o = s["file"] in landing_files, s["file"] in original_files
    where = ORIGINAL_DIR if in_o else (LANDING_DIR if in_l else None)
    if where is None:
        missing.append(s["file"])
        rows.append({"file": s["file"], "found_in": "-", "size_bytes": None, "check": "MISSING"})
    else:
        size = os.path.getsize(f"{where}/{s['file']}")
        note = "size matches inspection report" if size == s["exp_bytes"] else f"WARNING: size differs from inspection report ({s['exp_bytes']})"
        rows.append({"file": s["file"], "found_in": "original" if in_o else "landing", "size_bytes": size, "check": note})
display(pd.DataFrame(rows))

expected_names = {s["file"] for s in SOURCES}
unexpected = [f for f in landing_files if f not in expected_names]
if unexpected:
    print("Files in landing that are NOT expected (ignored):", unexpected)
if missing:
    raise FileNotFoundError(
        f"{len(missing)} file(s) missing: {missing}. Upload them to {LANDING_DIR} with the EXACT names "
        f"(remember: 'P  L March 2021.csv' has TWO spaces between P and L), then re-run this cell.")
print("All 7 source files found.")

file,found_in,size_bytes,check
Amazon Sale Report.csv,original,68923428,size matches inspection report
Sale Report.csv,original,433643,size matches inspection report
International sale Report.csv,original,3112676,size matches inspection report
May-2022.csv,original,126617,size matches inspection report
P L March 2021.csv,original,135938,size matches inspection report
Expense IIGF.csv,original,496,size matches inspection report
Cloud Warehouse Compersion Chart.csv,original,4528,size matches inspection report


All 7 source files found.


## Cell 5 – Preserve the originals: copy `landing/` → `original/` and verify with SHA-256   CORRECTED (replaces original Cell 5)
* A file already in `original/` is **never overwritten** (the copy uses exclusive-create mode, so it physically cannot overwrite).
* **Any** copy error, checksum mismatch, or `original/` ≠ `landing/` difference **stops the notebook** (raises an error). Nothing is ingested.
* `landing/` files are left untouched.

In [0]:
copy_results = {}
for s in SOURCES:
    name = s["file"]
    src, dst = f"{LANDING_DIR}/{name}", f"{ORIGINAL_DIR}/{name}"
    created_by_us = False
    try:
        if os.path.exists(dst):
            if os.path.exists(src):
                if sha256_file(src) == sha256_file(dst):
                    status = "ALREADY_PRESENT (identical to landing, left untouched)"
                else:
                    status = "FAILED: original/ differs from landing/ - original/ NOT overwritten"
            else:
                status = "ALREADY_PRESENT (landing copy not found, cannot compare; left untouched)"
        else:
            with open(src, "rb") as fin, open(dst, "xb") as fout:   # "xb" = fail if dst exists -> never overwrites
                created_by_us = True
                shutil.copyfileobj(fin, fout, 1024 * 1024)
            if sha256_file(src) == sha256_file(dst):
                status = "COPIED_VERIFIED (SHA-256 match)"
            else:
                os.remove(dst)                                       # removes only the bad copy created just now
                status = "FAILED: checksum mismatch after copy (bad copy removed)"
    except Exception as e:
        if created_by_us and os.path.exists(dst):
            os.remove(dst)                                           # partial copy created by this run only
        status = f"FAILED: {e}"
    copy_results[name] = {"status": status, "sha256": sha256_file(dst) if os.path.exists(dst) and not status.startswith("FAILED") else None}

display(pd.DataFrame([{"file": k, "status": v["status"], "sha256": v["sha256"]} for k, v in copy_results.items()]))
bad = [k for k, v in copy_results.items() if v["status"].startswith("FAILED")]
if bad:
    raise RuntimeError(f"STOP: copy/verification failed for {bad}. Nothing was ingested. Fix the cause, then re-run from Cell 4. "
                       f"(Existing files in original/ were not overwritten; if original/ is wrong, a human must decide what to do.)")
print("All 7 originals are in original/ and verified. Safe to continue.")

file,status,sha256
Amazon Sale Report.csv,"ALREADY_PRESENT (identical to landing, left untouched)",ac9a366da5f3af418831fa53da9eebfc3b3838803e3774eec97dbcb291441beb
Sale Report.csv,"ALREADY_PRESENT (identical to landing, left untouched)",60320c9831d0194c52ce5791f03f9c4f12e3474490a931d7cd23282290a28e6d
International sale Report.csv,"ALREADY_PRESENT (identical to landing, left untouched)",b0497693c317f456311f406a3a7d1cb15d38949143a50b6b15ab5c50c6e2baaf
May-2022.csv,"ALREADY_PRESENT (identical to landing, left untouched)",f24628666e406e96515a5beeb8cfc6e4f2350594217800b67f3613e0d98c32a7
P L March 2021.csv,"ALREADY_PRESENT (identical to landing, left untouched)",978679bb9d41e4b8a96dc5c65b0e0bae4b65546fa8a2f5f60457a93491811298
Expense IIGF.csv,"ALREADY_PRESENT (identical to landing, left untouched)",b1fa321a3aa963d91d06fc032a83848baec491cf13f9abd3ffbe75c6770c70db
Cloud Warehouse Compersion Chart.csv,"ALREADY_PRESENT (identical to landing, left untouched)",3c8e296064b7f3b98021f573c9e6777673234e67a9673ddc248995fb9fbb4c90


All 7 originals are in original/ and verified. Safe to continue.


## Cell 6 – Column-name mapping (for the 4 parsed files)  CORRECTED (replaces original Cell 6)
Delta tables work best with simple column names, so names are normalised (e.g. `Sales Channel ` → `sales_channel`, `Unnamed: 22` → `unnamed_22`).
**No column is dropped.** The original names (with their exact spaces) are saved in `_column_name_mapping`. A write error now stops the notebook.

In [0]:
COLMAP = {}          # file -> (original_names, bronze_names)
map_rows = []
for s in SOURCES:
    if s["kind"] != "csv":
        continue
    orig = read_csv_header(f"{ORIGINAL_DIR}/{s['file']}")
    new = to_bronze_names(orig)
    COLMAP[s["file"]] = (orig, new)
    for pos, (o, b) in enumerate(zip(orig, new), start=1):
        map_rows.append((s["file"], s["table"], pos, o, repr(o), b))

map_schema = StructType([StructField("source_file", StringType()), StructField("bronze_table", StringType()),
                         StructField("column_position", LongType()), StructField("original_name", StringType()),
                         StructField("original_name_repr", StringType()), StructField("bronze_name", StringType())])
MAP_TABLE = "_column_name_mapping"
try:
    map_stage = write_bronze_table(spark.createDataFrame(map_rows, map_schema), MAP_TABLE)
    map_count = spark.table(fqn(MAP_TABLE)).count()
except Exception as e:
    raise RuntimeError(f"Could not write/read {MAP_TABLE}: {e}")
if map_count != len(map_rows):
    msg = f"{MAP_TABLE} has {map_count} rows but {len(map_rows)} were expected"
    if map_stage == "SKIPPED":
        print("WARNING (existing table, not overwritten):", msg)
    else:
        raise RuntimeError(msg)
print(f"{MAP_TABLE}: {map_stage}  ({map_count} rows)" + ("  <- existing table, NOT newly written" if map_stage == "SKIPPED" else ""))
display(pd.DataFrame(map_rows, columns=["source_file", "bronze_table", "pos", "original_name", "original_repr", "bronze_name"])
        .query("original_name != bronze_name"))

_column_name_mapping: SKIPPED  (66 rows)  <- existing table, NOT newly written


source_file,bronze_table,pos,original_name,original_repr,bronze_name
Amazon Sale Report.csv,amazon_sale_report,2,Order ID,'Order ID',order_id
Amazon Sale Report.csv,amazon_sale_report,3,Date,'Date',date
Amazon Sale Report.csv,amazon_sale_report,4,Status,'Status',status
Amazon Sale Report.csv,amazon_sale_report,5,Fulfilment,'Fulfilment',fulfilment
Amazon Sale Report.csv,amazon_sale_report,6,Sales Channel,'Sales Channel ',sales_channel
Amazon Sale Report.csv,amazon_sale_report,7,ship-service-level,'ship-service-level',ship_service_level
Amazon Sale Report.csv,amazon_sale_report,8,Style,'Style',style
Amazon Sale Report.csv,amazon_sale_report,9,SKU,'SKU',sku
Amazon Sale Report.csv,amazon_sale_report,10,Category,'Category',category
Amazon Sale Report.csv,amazon_sale_report,11,Size,'Size',size


## Cell 7 – Ingest the 4 parsed CSV files (all columns as string)   CORRECTED (replaces original Cell 7)
Options: `inferSchema=false` (no typing), `FAILFAST` (a malformed row raises an error instead of being silently dropped),
whitespace is **not** trimmed, quotes are honoured. Nothing is filtered or de-duplicated.
Result per file: `CREATED`, `SKIPPED` (table already existed, untouched), `OVERWRITTEN` (only if the switch is True) or `FAILED`.

In [0]:
ingest_results = {}   # file -> {"status": CREATED | SKIPPED | OVERWRITTEN | FAILED, "error": ...}

def ingest_csv(s):
    if table_exists(s["table"]) and not OVERWRITE_EXISTING_TABLES:
        return "SKIPPED"
    orig, new = COLMAP[s["file"]]
    df = (spark.read.format("csv")
          .option("header", "true")
          .option("inferSchema", "false")                 # everything stays string
          .option("sep", ",").option("quote", '"').option("escape", '"')
          .option("multiLine", "true")                    # safe if a quoted cell ever contains a line break
          .option("encoding", "UTF-8")
          .option("mode", "FAILFAST")                     # fail loudly rather than lose/alter rows
          .option("ignoreLeadingWhiteSpace", "false")     # keep whitespace exactly as in the file
          .option("ignoreTrailingWhiteSpace", "false")
          .load(f"{ORIGINAL_DIR}/{s['file']}"))           # spaces in the filename are fine
    if len(df.columns) != len(new):
        raise ValueError(f"Spark found {len(df.columns)} columns but the header has {len(new)}.")
    df = add_metadata(df.toDF(*new), s["file"])           # rename by position -> no column is lost
    return write_bronze_table(df, s["table"])

for s in SOURCES:
    if s["kind"] != "csv":
        continue
    try:
        ingest_results[s["file"]] = {"status": ingest_csv(s), "error": None}
    except Exception as e:
        ingest_results[s["file"]] = {"status": "FAILED", "error": str(e)[:500]}
    r = ingest_results[s["file"]]
    print(f"{s['file']:40s} -> {s['table']:32s} {r['status']}", r["error"] or "")

Amazon Sale Report.csv                   -> amazon_sale_report               SKIPPED 
Sale Report.csv                          -> sale_report                      SKIPPED 
May-2022.csv                             -> may_2022                         SKIPPED 
P  L March 2021.csv                      -> pl_march_2021                    SKIPPED 


## Cell 8 – Ingest the 3 raw-line files (International, Expense IIGF, Cloud Warehouse)  CORRECTED (replaces original Cell 8)
Each physical line becomes one row: `line_number` (1 = the file's first line) and `raw_line` (the exact text).
Files are read in Python on the driver (3 MB, 496 B, 4.5 KB), so `line_number` is reliable. Splitting sub-tables is left to Silver.
Result per file: `CREATED`, `SKIPPED`, `OVERWRITTEN` or `FAILED` (same meaning as Cell 7).

In [0]:
RAW_SCHEMA = StructType([StructField("line_number", LongType(), False), StructField("raw_line", StringType(), True)])

def ingest_raw(s):
    if table_exists(s["table"]) and not OVERWRITE_EXISTING_TABLES:
        return "SKIPPED"
    lines, _ = read_raw_lines(f"{ORIGINAL_DIR}/{s['file']}")
    df = spark.createDataFrame([(i + 1, line) for i, line in enumerate(lines)], schema=RAW_SCHEMA)
    return write_bronze_table(add_metadata(df, s["file"]), s["table"])

for s in SOURCES:
    if s["kind"] != "raw":
        continue
    try:
        ingest_results[s["file"]] = {"status": ingest_raw(s), "error": None}
    except Exception as e:
        ingest_results[s["file"]] = {"status": "FAILED", "error": str(e)[:500]}
    r = ingest_results[s["file"]]
    print(f"{s['file']:40s} -> {s['table']:32s} {r['status']}", r["error"] or "")

International sale Report.csv            -> international_sale_report_raw    SKIPPED 
Expense IIGF.csv                         -> expense_iigf_raw                 SKIPPED 
Cloud Warehouse Compersion Chart.csv     -> cloud_warehouse_comparison_raw   SKIPPED 


## Cell 9 – Validation (source file vs Bronze table)  CORRECTED (replaces original Cell 9)
Computed independently of Spark's CSV reader. Every table is checked, **including tables that were SKIPPED**, but a skipped table is never reported as newly ingested.
* **Parsed tables:** row count, column count + column names, non-empty cells in **every column**, and 0 malformed (ragged) rows in the source.
* **Raw-line tables:** line count, `line_number` = 1..N with no gaps/duplicates, no NULL lines, and rebuilding the file gives the **same SHA-256** as the original.
* Source counts are also compared with the inspection report (shown, not a failure).

Final status: `CREATED_VALIDATED` · `OVERWRITTEN_VALIDATED` · `SKIPPED_EXISTING` (existing table matches source, NOT newly ingested) · `SKIPPED_EXISTING_MISMATCH` · `FAILED`

In [0]:
def final_status(stage, checks):
    if stage not in ("CREATED", "OVERWRITTEN", "SKIPPED") or checks == "ERROR":
        return "FAILED"
    if stage == "SKIPPED":
        return "SKIPPED_EXISTING" if checks == "MATCH" else "SKIPPED_EXISTING_MISMATCH"
    return f"{stage}_VALIDATED" if checks == "MATCH" else "FAILED"

validation = {}
for s in SOURCES:
    name, table = s["file"], s["table"]
    stage = ingest_results.get(name, {}).get("status", "NOT_RUN")
    v = {"stage": stage, "src_count": None, "bronze_count": None, "checks": None, "matches_report": None, "details": []}
    try:
        path = f"{ORIGINAL_DIR}/{name}"
        if stage not in ("CREATED", "OVERWRITTEN", "SKIPPED"):
            v["checks"] = "ERROR"
            v["details"].append(f"ingestion stage = {stage}: {ingest_results.get(name, {}).get('error')}")
        elif not table_exists(table):
            v["checks"] = "MISMATCH"
            v["details"].append("Bronze table does not exist")
        else:
            bdf = spark.table(fqn(table))
            v["bronze_count"] = bdf.count()
            problems = []
            if s["kind"] == "csv":
                header, n, src_nonempty, ragged = csv_source_profile(path)
                v["src_count"] = n
                new = COLMAP[name][1]
                if ragged:
                    problems.append(f"{ragged} malformed row(s) in the source (field count != header)")
                if v["bronze_count"] != n:
                    problems.append(f"row count source={n} bronze={v['bronze_count']}")
                if len(header) != len(new):
                    problems.append(f"source header has {len(header)} columns, mapping has {len(new)}")
                if bdf.columns != new + META_COLS:
                    problems.append(f"column names/count differ: bronze={len(bdf.columns)} expected={len(new) + len(META_COLS)}")
                else:
                    aggs = [F.sum(F.when(F.col(c).isNotNull() & (F.col(c) != ""), 1).otherwise(0)).alias(c) for c in new]
                    got = bdf.agg(*aggs).collect()[0]
                    for c, exp in zip(new, src_nonempty):
                        if int(got[c] or 0) != exp:
                            problems.append(f"column '{c}': non-empty cells source={exp} bronze={int(got[c] or 0)}")
            else:
                lines, ends_nl = read_raw_lines(path)
                v["src_count"] = len(lines)
                if v["bronze_count"] != len(lines):
                    problems.append(f"line count source={len(lines)} bronze={v['bronze_count']}")
                st = bdf.agg(F.min("line_number").alias("mn"), F.max("line_number").alias("mx"),
                             F.countDistinct("line_number").alias("dn"),
                             F.sum(F.when(F.col("raw_line").isNull(), 1).otherwise(0)).alias("nulls")).collect()[0]
                if not (st["mn"] == 1 and st["mx"] == len(lines) and st["dn"] == len(lines)):
                    problems.append(f"line_number is not exactly 1..{len(lines)} (min={st['mn']}, max={st['mx']}, distinct={st['dn']})")
                if int(st["nulls"] or 0) > 0:
                    problems.append(f"{st['nulls']} NULL raw_line value(s)")
                rebuilt = [r["raw_line"] for r in bdf.orderBy("line_number").select("raw_line").collect()]
                text = "\n".join("" if x is None else x for x in rebuilt) + ("\n" if ends_nl else "")
                if hashlib.sha256(text.encode("utf-8")).hexdigest() != sha256_file(path):
                    problems.append("rebuilt file does NOT match the original (SHA-256 differs)")
            v["matches_report"] = (v["src_count"] == s["exp_count"])
            if not v["matches_report"]:
                v["details"].append(f"note: source count {v['src_count']} differs from inspection report ({s['exp_count']})")
            v["details"] += problems
            v["checks"] = "MATCH" if not problems else "MISMATCH"
    except Exception as e:
        v["checks"] = "ERROR"
        v["details"].append(str(e)[:500])
    v["final"] = final_status(v["stage"], v["checks"])
    validation[name] = v
    print(f"[{v['final']}] {name}: source={v['src_count']} bronze={v['bronze_count']} (stage={v['stage']}, checks={v['checks']})")
    for d in v["details"]:
        print("     -", d)

[SKIPPED_EXISTING] Amazon Sale Report.csv: source=128975 bronze=128975 (stage=SKIPPED, checks=MATCH)
[SKIPPED_EXISTING] Sale Report.csv: source=9271 bronze=9271 (stage=SKIPPED, checks=MATCH)
[SKIPPED_EXISTING] International sale Report.csv: source=37433 bronze=37433 (stage=SKIPPED, checks=MATCH)
[SKIPPED_EXISTING] May-2022.csv: source=1330 bronze=1330 (stage=SKIPPED, checks=MATCH)
[SKIPPED_EXISTING] P  L March 2021.csv: source=1330 bronze=1330 (stage=SKIPPED, checks=MATCH)
[SKIPPED_EXISTING] Expense IIGF.csv: source=18 bronze=18 (stage=SKIPPED, checks=MATCH)
[SKIPPED_EXISTING] Cloud Warehouse Compersion Chart.csv: source=51 bronze=51 (stage=SKIPPED, checks=MATCH)


## Cell 10 – Final summary (and an append-only run log) CORRECTED (replaces original Cell 10)
The notebook ends with an **error** if any table is `FAILED` or `SKIPPED_EXISTING_MISMATCH`, so a failure cannot be missed.

In [0]:
summary_rows = []
for s in SOURCES:
    n, v = s["file"], validation[s["file"]]
    summary_rows.append({
        "Source file": n,
        "Bronze table": f"{BRONZE_SCHEMA}.{s['table']}",
        "Ingestion method": METHOD_TEXT[s["kind"]],
        "Count basis": COUNT_BASIS[s["kind"]],
        "Source count": v["src_count"],
        "Bronze count": v["bronze_count"],
        "Matches inspection report": v["matches_report"],
        "Original copy": copy_results.get(n, {}).get("status"),
        "Original SHA-256": copy_results.get(n, {}).get("sha256"),
        "Stage": v["stage"],
        "Checks": v["checks"],
        "Final status": v["final"],
    })
summary_df = pd.DataFrame(summary_rows)
display(summary_df)

counts = summary_df["Final status"].value_counts().to_dict()
print("\nResult:", counts)
print(f"Mapping table {MAP_TABLE}: {map_stage}" + (" (existing, not newly written)" if map_stage == "SKIPPED" else ""))
n_new = sum(1 for r in summary_rows if r["Final status"] in ("CREATED_VALIDATED", "OVERWRITTEN_VALIDATED"))
n_skip = sum(1 for r in summary_rows if r["Final status"] == "SKIPPED_EXISTING")
bad_rows = [r["Source file"] for r in summary_rows if r["Final status"] in ("FAILED", "SKIPPED_EXISTING_MISMATCH")]
print(f"Newly ingested and validated: {n_new} | Skipped (already existed, not re-ingested): {n_skip} | Problems: {len(bad_rows)}")

# Append-only log (never overwrites earlier runs). A failure here does not affect the Bronze tables.
try:
    log_schema = StructType([StructField("batch_id", StringType()), StructField("run_timestamp", TimestampType()),
                             StructField("source_file", StringType()), StructField("bronze_table", StringType()),
                             StructField("ingestion_method", StringType()), StructField("count_basis", StringType()),
                             StructField("source_count", LongType()), StructField("bronze_count", LongType()),
                             StructField("table_status", StringType()), StructField("validation", StringType())])
    log_rows = [(BATCH_ID, INGESTION_TS, r["Source file"], r["Bronze table"], r["Ingestion method"], r["Count basis"],
                 r["Source count"], r["Bronze count"], r["Final status"], r["Checks"]) for r in summary_rows]
    spark.createDataFrame(log_rows, log_schema).write.format("delta").mode("append").saveAsTable(fqn("_ingestion_log"))
    print("Run appended to", f"{BRONZE_SCHEMA}._ingestion_log")
except Exception as e:
    print("Could not write _ingestion_log (Bronze tables are not affected):", str(e)[:300])

if bad_rows:
    raise RuntimeError(f"Bronze ingestion NOT successful for: {bad_rows}. Read the messages in Cells 5, 7, 8 and 9.")

Source file,Bronze table,Ingestion method,Count basis,Source count,Bronze count,Matches inspection report,Original copy,Original SHA-256,Stage,Checks,Final status
Amazon Sale Report.csv,bronze.amazon_sale_report,"CSV parsed, all columns as string",data rows (excl. header),128975,128975,true,"ALREADY_PRESENT (identical to landing, left untouched)",ac9a366da5f3af418831fa53da9eebfc3b3838803e3774eec97dbcb291441beb,SKIPPED,MATCH,SKIPPED_EXISTING
Sale Report.csv,bronze.sale_report,"CSV parsed, all columns as string",data rows (excl. header),9271,9271,true,"ALREADY_PRESENT (identical to landing, left untouched)",60320c9831d0194c52ce5791f03f9c4f12e3474490a931d7cd23282290a28e6d,SKIPPED,MATCH,SKIPPED_EXISTING
International sale Report.csv,bronze.international_sale_report_raw,"Raw lines (line_number, raw_line)",physical lines (incl. header),37433,37433,true,"ALREADY_PRESENT (identical to landing, left untouched)",b0497693c317f456311f406a3a7d1cb15d38949143a50b6b15ab5c50c6e2baaf,SKIPPED,MATCH,SKIPPED_EXISTING
May-2022.csv,bronze.may_2022,"CSV parsed, all columns as string",data rows (excl. header),1330,1330,true,"ALREADY_PRESENT (identical to landing, left untouched)",f24628666e406e96515a5beeb8cfc6e4f2350594217800b67f3613e0d98c32a7,SKIPPED,MATCH,SKIPPED_EXISTING
P L March 2021.csv,bronze.pl_march_2021,"CSV parsed, all columns as string",data rows (excl. header),1330,1330,true,"ALREADY_PRESENT (identical to landing, left untouched)",978679bb9d41e4b8a96dc5c65b0e0bae4b65546fa8a2f5f60457a93491811298,SKIPPED,MATCH,SKIPPED_EXISTING
Expense IIGF.csv,bronze.expense_iigf_raw,"Raw lines (line_number, raw_line)",physical lines (incl. header),18,18,true,"ALREADY_PRESENT (identical to landing, left untouched)",b1fa321a3aa963d91d06fc032a83848baec491cf13f9abd3ffbe75c6770c70db,SKIPPED,MATCH,SKIPPED_EXISTING
Cloud Warehouse Compersion Chart.csv,bronze.cloud_warehouse_comparison_raw,"Raw lines (line_number, raw_line)",physical lines (incl. header),51,51,true,"ALREADY_PRESENT (identical to landing, left untouched)",3c8e296064b7f3b98021f573c9e6777673234e67a9673ddc248995fb9fbb4c90,SKIPPED,MATCH,SKIPPED_EXISTING



Result: {'SKIPPED_EXISTING': 7}
Mapping table _column_name_mapping: SKIPPED (existing, not newly written)
Newly ingested and validated: 0 | Skipped (already existed, not re-ingested): 7 | Problems: 0
Run appended to bronze._ingestion_log


# Silver and Gold Layers – E-Commerce Sales Intelligence

These cells extend the completed Bronze notebook. They do not modify or overwrite Bronze tables.

**Silver:** standardized, typed, quality-checked Delta tables in `workspace.silver`.
**Gold:** aggregated, analytics-ready Delta tables in `workspace.gold`.

The four parsed CSV sources have known columns from `_column_name_mapping`. The three irregular files remain line-preserved in Silver because their internal layouts have not yet been formally defined; do not guess their structure.


In [0]:
# Cell 11 – Silver/Gold configuration and schemas
CATALOG = "workspace"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{SILVER_SCHEMA}`")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{GOLD_SCHEMA}`")

def silver_fqn(table):
    return f"`{CATALOG}`.`{SILVER_SCHEMA}`.`{table}`"

def gold_fqn(table):
    return f"`{CATALOG}`.`{GOLD_SCHEMA}`.`{table}`"

print("Silver and Gold schemas are ready.")


Silver and Gold schemas are ready.


## Cell 12 – Build Silver tables

Cleaning rules used here:
- Trim surrounding whitespace in string columns and convert empty strings to NULL.
- Preserve the original Bronze columns and metadata.
- Add explicitly typed helper columns rather than silently replacing source strings.
- Remove exact duplicate rows only (not business-key duplicates).
- Keep raw-line sources unchanged until their document/table layouts are profiled.


In [0]:
CATALOG = "workspace"

In [0]:
# Catalog configuration
CATALOG = "workspace"

# Bronze table location
def bronze_fqn(table):
    return f"{CATALOG}.bronze.{table}"

# Silver table location
def silver_fqn(table):
    return f"{CATALOG}.silver.{table}"

# Gold table location
def gold_fqn(table):
    return f"{CATALOG}.gold.{table}"

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.gold")

print("Silver and Gold schemas are ready!")

Silver and Gold schemas are ready!


In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

def clean_strings(df):
    # Trim text values and turn empty strings into NULL; leave non-string columns unchanged.
    out = df
    for field in df.schema.fields:
        if isinstance(field.dataType, T.StringType):
            out = out.withColumn(
                field.name,
                F.when(F.trim(F.col(field.name)) == "", F.lit(None).cast("string"))
                 .otherwise(F.trim(F.col(field.name)))
            )
    return out

def save_delta(df, full_name):
    (df.write.format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(full_name))
    print(f"Saved {full_name}: {df.count():,} rows")

# 12A. Amazon sales: retain all source fields, add typed analytical fields.
amazon_b = spark.table(f"`{CATALOG}`.`bronze`.`amazon_sale_report`")
amazon_s = clean_strings(amazon_b).dropDuplicates()

# Source date values can vary; try common date patterns without discarding unparseable values.
date_text = F.col("date")
amazon_s = (amazon_s
    .withColumn("order_date",
        F.coalesce(
            F.to_date(date_text, "M-d-yy"),
            F.to_date(date_text, "MM-dd-yy"),
            F.to_date(date_text, "yyyy-MM-dd"),
            F.to_date(date_text, "dd-MM-yyyy"),
            F.to_date(date_text, "dd/MM/yyyy"),
            F.to_date(date_text, "M/d/yyyy")
        ))
    .withColumn("qty_int", F.expr("try_cast(qty as INT)"))
    .withColumn("amount_decimal", F.expr("try_cast(amount as DECIMAL(18,2))"))
    .withColumn("b2b_bool",
        F.when(F.lower(F.col("b2b")).isin("true", "1", "yes", "y"), F.lit(True))
         .when(F.lower(F.col("b2b")).isin("false", "0", "no", "n"), F.lit(False))
         .otherwise(F.lit(None).cast("boolean")))
)
save_delta(amazon_s, silver_fqn("amazon_sale_report"))

# 12B. Sale Report: stock is typed as integer; all source columns retained.
sale_b = spark.table(f"`{CATALOG}`.`bronze`.`sale_report`")
sale_s = (clean_strings(sale_b).dropDuplicates()
          .withColumn("stock_int", F.expr("try_cast(stock as INT)")))
save_delta(sale_s, silver_fqn("sale_report"))

# 12C. May-2022 catalogue/pricing file: add decimal helper columns.
may_b = spark.table(f"`{CATALOG}`.`bronze`.`may_2022`")
may_s = clean_strings(may_b).dropDuplicates()
for c in ["weight", "tp", "mrp_old", "final_mrp_old", "ajio_mrp",
          "amazon_mrp", "amazon_fba_mrp", "flipkart_mrp", "limeroad_mrp",
          "myntra_mrp", "paytm_mrp", "snapdeal_mrp"]:
    may_s = may_s.withColumn(f"{c}_decimal", F.expr(f"try_cast(`{c}` as DECIMAL(18,2))"))
save_delta(may_s, silver_fqn("may_2022"))

# 12D. P&L March 2021 catalogue/pricing file: add decimal helper columns.
pl_b = spark.table(f"`{CATALOG}`.`bronze`.`pl_march_2021`")
pl_s = clean_strings(pl_b).dropDuplicates()
for c in ["weight", "tp_1", "tp_2", "mrp_old", "final_mrp_old", "ajio_mrp",
          "amazon_mrp", "amazon_fba_mrp", "flipkart_mrp", "limeroad_mrp",
          "myntra_mrp", "paytm_mrp", "snapdeal_mrp"]:
    pl_s = pl_s.withColumn(f"{c}_decimal", F.expr(f"try_cast(`{c}` as DECIMAL(18,2))"))
save_delta(pl_s, silver_fqn("pl_march_2021"))

# 12E–12G. Irregular documents: preserve exact line content and order.
# Parsing into business columns is intentionally deferred until each file's sections are profiled.
for table in ["international_sale_report_raw", "expense_iigf_raw", "cloud_warehouse_comparison_raw"]:
    raw_df = spark.table(f"`{CATALOG}`.`bronze`.`{table}`")
    save_delta(raw_df, silver_fqn(table))


Saved workspace.silver.amazon_sale_report: 128,975 rows
Saved workspace.silver.sale_report: 9,271 rows
Saved workspace.silver.may_2022: 1,330 rows
Saved workspace.silver.pl_march_2021: 1,330 rows
Saved workspace.silver.international_sale_report_raw: 37,433 rows
Saved workspace.silver.expense_iigf_raw: 18 rows
Saved workspace.silver.cloud_warehouse_comparison_raw: 51 rows


## Cell 13 – Silver data-quality checks

This creates a compact quality report. Invalid casts and unparsed dates are counted, not silently removed. Review the report before relying on those fields in business reporting.


In [0]:
amazon_s = spark.table(silver_fqn("amazon_sale_report"))
sale_s = spark.table(silver_fqn("sale_report"))
may_s = spark.table(silver_fqn("may_2022"))
pl_s = spark.table(silver_fqn("pl_march_2021"))

quality_rows = [
    ("amazon_sale_report", "rows", amazon_s.count()),
    ("amazon_sale_report", "null_order_id", amazon_s.filter(F.col("order_id").isNull()).count()),
    ("amazon_sale_report", "unparsed_order_date", amazon_s.filter(F.col("date").isNotNull() & F.col("order_date").isNull()).count()),
    ("amazon_sale_report", "invalid_qty_cast", amazon_s.filter(F.col("qty").isNotNull() & F.col("qty_int").isNull()).count()),
    ("amazon_sale_report", "invalid_amount_cast", amazon_s.filter(F.col("amount").isNotNull() & F.col("amount_decimal").isNull()).count()),
    ("sale_report", "rows", sale_s.count()),
    ("sale_report", "invalid_stock_cast", sale_s.filter(F.col("stock").isNotNull() & F.col("stock_int").isNull()).count()),
    ("may_2022", "rows", may_s.count()),
    ("pl_march_2021", "rows", pl_s.count()),
]
quality_df = spark.createDataFrame(quality_rows, ["table_name", "check_name", "result_count"])
display(quality_df.orderBy("table_name", "check_name"))
save_delta(quality_df, silver_fqn("data_quality_summary"))


table_name,check_name,result_count
amazon_sale_report,invalid_amount_cast,0
amazon_sale_report,invalid_qty_cast,0
amazon_sale_report,null_order_id,0
amazon_sale_report,rows,128975
amazon_sale_report,unparsed_order_date,0
may_2022,rows,1330
pl_march_2021,rows,1330
sale_report,invalid_stock_cast,9235
sale_report,rows,9271


Saved workspace.silver.data_quality_summary: 9 rows


## Cell 14 – Build Gold analytics tables

Gold metrics below are based only on the Amazon sales source, because it contains order, date, quantity, amount, category, fulfilment and shipping fields. Amount is treated as the source's reported amount; confirm its business meaning/currency with the data owner before external financial reporting.


In [0]:
amazon_s = spark.table(silver_fqn("amazon_sale_report"))

# Use rows with a parsed date and numeric amount for time-based revenue metrics.
valid_sales = amazon_s.filter(
    F.col("order_date").isNotNull() & F.col("amount_decimal").isNotNull()
)

# Gold 1: daily sales performance.
gold_daily = (valid_sales
    .groupBy("order_date")
    .agg(
        F.countDistinct("order_id").alias("order_count"),
        F.sum(F.coalesce(F.col("qty_int"), F.lit(0))).alias("units_sold"),
        F.round(F.sum("amount_decimal"), 2).alias("reported_sales_amount"),
        F.count("*").alias("line_count")
    )
    .orderBy("order_date"))
save_delta(gold_daily, gold_fqn("daily_sales_summary"))

# Gold 2: category performance.
gold_category = (valid_sales
    .groupBy("category")
    .agg(
        F.countDistinct("order_id").alias("order_count"),
        F.sum(F.coalesce(F.col("qty_int"), F.lit(0))).alias("units_sold"),
        F.round(F.sum("amount_decimal"), 2).alias("reported_sales_amount")
    )
    .orderBy(F.desc("reported_sales_amount")))
save_delta(gold_category, gold_fqn("category_sales_summary"))

# Gold 3: fulfilment performance.
gold_fulfilment = (valid_sales
    .groupBy("fulfilment")
    .agg(
        F.countDistinct("order_id").alias("order_count"),
        F.sum(F.coalesce(F.col("qty_int"), F.lit(0))).alias("units_sold"),
        F.round(F.sum("amount_decimal"), 2).alias("reported_sales_amount")
    )
    .orderBy(F.desc("reported_sales_amount")))
save_delta(gold_fulfilment, gold_fqn("fulfilment_sales_summary"))

# Gold 4: state-wise sales.
gold_state = (valid_sales
    .groupBy("ship_state")
    .agg(
        F.countDistinct("order_id").alias("order_count"),
        F.sum(F.coalesce(F.col("qty_int"), F.lit(0))).alias("units_sold"),
        F.round(F.sum("amount_decimal"), 2).alias("reported_sales_amount")
    )
    .orderBy(F.desc("reported_sales_amount")))
save_delta(gold_state, gold_fqn("state_sales_summary"))

# Gold 5: SKU performance.
gold_sku = (valid_sales
    .groupBy("sku", "style", "category")
    .agg(
        F.countDistinct("order_id").alias("order_count"),
        F.sum(F.coalesce(F.col("qty_int"), F.lit(0))).alias("units_sold"),
        F.round(F.sum("amount_decimal"), 2).alias("reported_sales_amount")
    )
    .orderBy(F.desc("reported_sales_amount")))
save_delta(gold_sku, gold_fqn("sku_sales_summary"))


Saved workspace.gold.daily_sales_summary: 91 rows
Saved workspace.gold.category_sales_summary: 9 rows
Saved workspace.gold.fulfilment_sales_summary: 2 rows
Saved workspace.gold.state_sales_summary: 69 rows
Saved workspace.gold.sku_sales_summary: 7,157 rows


## Cell 15 – Final Silver/Gold validation

Confirm each target table exists and inspect row counts. This does not replace business validation of metric definitions.


In [0]:
silver_tables = [
    "amazon_sale_report", "sale_report", "may_2022", "pl_march_2021",
    "international_sale_report_raw", "expense_iigf_raw",
    "cloud_warehouse_comparison_raw", "data_quality_summary"
]
gold_tables = [
    "daily_sales_summary", "category_sales_summary",
    "fulfilment_sales_summary", "state_sales_summary", "sku_sales_summary"
]

validation_rows = []
for t in silver_tables:
    df = spark.table(silver_fqn(t))
    validation_rows.append(("silver", t, df.count(), len(df.columns)))
for t in gold_tables:
    df = spark.table(gold_fqn(t))
    validation_rows.append(("gold", t, df.count(), len(df.columns)))

layer_validation = spark.createDataFrame(
    validation_rows, ["layer", "table_name", "row_count", "column_count"]
)
display(layer_validation.orderBy("layer", "table_name"))

print("Silver and Gold tables created. Review data_quality_summary and confirm metric definitions before dashboard publication.")


layer,table_name,row_count,column_count
gold,category_sales_summary,9,4
gold,daily_sales_summary,91,5
gold,fulfilment_sales_summary,2,4
gold,sku_sales_summary,7157,6
gold,state_sales_summary,69,4
silver,amazon_sale_report,128975,31
silver,cloud_warehouse_comparison_raw,51,5
silver,data_quality_summary,9,3
silver,expense_iigf_raw,18,5
silver,international_sale_report_raw,37433,5


Silver and Gold tables created. Review data_quality_summary and confirm metric definitions before dashboard publication.
